In [74]:
from tornado import gen

csv_path = "/home/theo/PycharmProjects/Bookoflife/Original/data/raw/data.csv"
cdb_path = "/home/theo/PycharmProjects/Bookoflife/Original/data/raw/data.cbd"


In [75]:
import re
import pandas as pd
from pathlib import Path

def parse_cdb(cdb_path):
    """Parse NLSY79 codebook (.cdb) - simple version"""
    codebook = {}
    text = Path(cdb_path).read_text()

    # Split by long dashes
    blocks = re.split(r"-{40,}", text)

    for block in blocks:
        # Match header: H00016.00    [H40-BPAR-1]    Survey Year: XRND
        m = re.match(r"\s*(\w+\.\d+)\s+\[([^\]]+)\]\s+Survey Year:\s*(\S+)", block)
        if not m:
            continue

        rnum = m.group(1).replace(".", "")  # H0001600
        qname = m.group(2)                   # H40-BPAR-1
        year = m.group(3)                    # XRND

        # Extract question (after "PRIMARY VARIABLE")
        question = ""
        qm = re.search(r"PRIMARY VARIABLE\s*\n\s*\n\s*(.+?)(?:\n\s*\n|$)", block, re.DOTALL)
        if qm:
            question = qm.group(1).strip().replace("\n", " ")

        codebook[rnum] = {
            "rnum": rnum,
            "qname": qname,
            "year": year,
            "question": question,
        }

    return codebook


In [76]:
cdb_path = "/home/theo/PycharmProjects/Bookoflife/Original/data/raw/data.cdb"
codebook = parse_cdb(cdb_path)

cb_df = pd.DataFrame(codebook.values())


In [77]:
invalid_year = ~cb_df["year"].astype(str).str.fullmatch(r"\d{4}")

unique_qname = cb_df.groupby("qname")["qname"].transform("size") == 1

# retroperspecitv questions
living_struc = (
    cb_df["year"].astype(str).eq("1988")
    & cb_df["question"]
        .astype(str)
        .str.lower()
        .str.startswith(("lived with", "lived in"), na=False)
)

##sind die ganzen 1988 struktur variablen auch drinnen -> wie identifizieren wir die? questinon text ist mit lived with..
cbd_p = cb_df[unique_qname & ~living_struc & ~(cb_df["year"] == "XRND")] #XRND variablen raus da kein informationsverlust
cbd_pl = cb_df[~(invalid_year | unique_qname) | (cb_df["rnum"] == "R0000100")]
cbd_ls = cb_df[living_struc | (cb_df["rnum"] == "R0000100")]

In [78]:
print(len(cbd_p))
print(len(cbd_pl))
print(len(cbd_ls))


25
181
255


In [104]:
df_csv = pd.read_csv(csv_path)
csv_pl = df_csv[cbd_pl["rnum"]]
csv_p = df_csv[cbd_p["rnum"]]
csv_ls = df_csv[cbd_ls["rnum"]]

In [105]:
#variables we want to make to long
rnum_vars_p = cbd_p["rnum"].tolist()
print(len(rnum_vars_p))
# melt: von wide zu long. id_vars ist  'R0000100'
melted_p = csv_p.melt(id_vars=["R0000100"], value_vars=rnum_vars_p,
                     var_name="rnum", value_name="value")


melted_p = melted_p.merge(cb_df[["rnum", "qname", "year", "question"]],
                      on="rnum", how="left")

melted_p = melted_p.rename(columns={"R0000100": "caseid"})
melted_p = melted_p.sort_values(["caseid", "rnum"]).reset_index(drop=True)

melted_p["syear"] = pd.to_numeric(melted_p["year"], errors="coerce")


25


In [106]:
pivoted_p = melted_p.pivot_table(
    index=['caseid', 'year'],
    columns='qname',
    values='value',
).reset_index()

In [121]:
pivoted_p["birthyear"] =  1979 - pivoted_p["FAM-1B"]
pivoted_p["year"] = pivoted_p["year"].replace("78SCRN", 1979) #replace da sonst strange jahre als str zu verwenden
pivoted_p["year"] = pivoted_p["year"].astype(int)

In [122]:
pivoted_p

qname,caseid,year,CRES-4,CRES-7,FAM-10,FAM-15,FAM-15A,FAM-1B,FAM-27C,FAM-8,...,Q9-89.10,Q9-90.04,Q9-91.07,Q9-91.08,Q9-91.10,Q9-92.02,SAMPLE_ID,SAMPLE_RACE,SAMPLE_SEX,birthyear
0,1,1979,NaN,NaN,-4.0,2.0,-4.0,20.0,-4.0,-4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,2.0,1959.0
1,1,1983,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,1987,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,1988,-5.0,-5.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,1996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,-5.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101483,12686,1988,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
101484,12686,1996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,-5.0,NaN,NaN,NaN,NaN
101485,12686,2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,-5.0,-5.0,-5.0,NaN,NaN,NaN,NaN,NaN,NaN
101486,12686,2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-5.0,NaN,NaN,NaN,-5.0,NaN,NaN,NaN,NaN,NaN


In [123]:
rnum_vars = cbd_pl["rnum"].tolist()
print(len(rnum_vars))
melted = csv_pl.melt(id_vars=["R0000100"], value_vars=rnum_vars,
                     var_name="rnum", value_name="value")

melted = melted.merge(cb_df[["rnum", "qname", "year", "question"]],
                      on="rnum", how="left")


melted = melted.rename(columns={"R0000100": "caseid"})
melted = melted.sort_values(["caseid", "rnum"]).reset_index(drop=True)


melted["syear"] = pd.to_numeric(melted["year"], errors="coerce")


181


In [124]:
pivoted_pl = melted.pivot_table(
    index=['caseid', 'year'],
    columns='qname',
    values='value',
).reset_index()


In [125]:
rnum_vars_ls = cbd_ls["rnum"].tolist()

# melt: von wide zu long. id_vars ist  'R0000100'
melted_ls = csv_ls.melt(id_vars=["R0000100"], value_vars=rnum_vars_ls,
                     var_name="rnum", value_name="value")

melted_ls = melted_ls.merge(cb_df[["rnum", "qname", "year", "question"]],
                      on="rnum", how="left")

melted_ls = melted_ls.rename(columns={"R0000100": "caseid"})
melted_ls = melted_ls.sort_values(["caseid", "rnum"]).reset_index(drop=True)

melted_ls["syear"] = pd.to_numeric(melted_ls["year"], errors="coerce")

In [126]:
pivoted_ls = melted_ls.pivot_table(
    index=['caseid', 'year'],
    columns='qname',
    values='value',
).reset_index()

In [127]:
def reshape_retrospective(df, var_prefix):
    df = df.dropna(subset=['birthyear'])

    cols = []
    for col in df.columns:
        if col.startswith(var_prefix.upper()):
            # Extrahiere die Zahl am Ende
            match = re.search(r'(\d+)$', col)
            if match:
                age_num = int(match.group(1))
                if 0 <= age_num <= 18:
                    cols.append(col)
    if not cols:
        print(f"Keine passenden Spalten gefunden!")
        return pd.DataFrame()

    long = df.melt(
        id_vars=['caseid', 'year', 'birthyear'],
        value_vars=cols,
        var_name='suffix',
        value_name=var_prefix
    )

    long['age'] = long['suffix'].str.extract(r'(\d+)$').astype(int)
    long['year'] = long['birthyear'] + long['age']  # ✓ direkt als 'year'

    return long[['caseid', 'year', 'age', 'birthyear', var_prefix]].sort_values(['caseid', 'year']).reset_index(drop=True)

In [128]:
# Merge mit caseid  year
pivoted_ls_with_birth = pivoted_ls.merge(
    pivoted_p[['caseid', 'birthyear']],  # Nur eindeutige caseid-birthyear
    on='caseid',
    how='left'
)
# Anwenden
adopt_long = reshape_retrospective(pivoted_ls_with_birth, 'adoptd')


In [129]:
cols_ending_18 = [col for col in pivoted_ls_with_birth.columns
                  if re.search(r'18$', col)]
prefixes = [re.sub(r'\d+$', '', col) for col in cols_ending_18]

long_dfs = [reshape_retrospective(pivoted_ls_with_birth, prefix) for prefix in prefixes]

result_ls = pd.concat(long_dfs, axis=1)
result_ls = result_ls.loc[:, ~result_ls.columns.duplicated()].sort_values(['caseid', 'year']).reset_index(drop=True)


In [130]:
result_ls['year'] = result_ls['year'].astype(int)
pivoted_pl['year'] = pivoted_pl['year'].astype(int)

merged_1 = result_ls.merge(pivoted_pl, on=['caseid', 'year'], how='outer')

merged_1['birthyear'] = merged_1.groupby('caseid')['birthyear'].transform(
    lambda x: x.fillna(x.dropna().iloc[0] if len(x.dropna()) > 0 else None))
merged_1
merged_1['age'] = pd.to_numeric(merged_1['year'], errors='coerce') - merged_1['birthyear'].astype(float)



In [133]:
merged_1

,caseid,year,age,birthyear,ADOPTD,ADOPTM,BDAD,BMOM,CHLDHM,DTENT,...,Q9-91.03,Q9-91.04,Q9-91.05,Q9-91.06,Q9-91.09,Q9-91_1,Q9-91_2,Q9-92.01,Q9-92_1,Q9-92_2
0,1,1959,0.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1960,1.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,1961,2.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,1962,3.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,1963,4.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
507323,12686,2010,49.0,1961.0,NaN,NaN,NaN,NaN,NaN,NaN,...,-5.0,NaN,-5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
507324,12686,2012,51.0,1961.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
507325,12686,2014,53.0,1961.0,NaN,NaN,NaN,NaN,NaN,NaN,...,-5.0,-5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
507326,12686,2016,55.0,1961.0,NaN,NaN,NaN,NaN,NaN,NaN,...,-5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [140]:
# Merge mit pivoted_p
+ = merged_1.merge(pivoted_p, on=['caseid', 'year'], how='outer')
merged_1['birthyear'] = merged_1.groupby('caseid')['birthyear'].transform(
    lambda x: x.fillna(x.dropna().iloc[0] if len(x.dropna()) > 0 else None))
final_result = final_result.sort_values(['caseid', 'year']).reset_index(drop=True)
final_result = final_result.rename(columns={"birthyear_x": "birthyear"})
final_result["age"] = final_result["year"] - final_result["birthyear"]


In [141]:
final_result

,caseid,year,age,birthyear,ADOPTD,ADOPTM,BDAD,BMOM,CHLDHM,DTENT,...,Q9-89.10,Q9-90.04,Q9-91.07,Q9-91.08,Q9-91.10,Q9-92.02,SAMPLE_ID,SAMPLE_RACE,SAMPLE_SEX,birthyear_y
0,1,1959,0.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1960,1.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,1961,2.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,1962,3.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,1963,4.0,1959.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522852,12686,2010,49.0,1961.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
522853,12686,2012,51.0,1961.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
522854,12686,2014,53.0,1961.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
522855,12686,2016,55.0,1961.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
